# Oracle upper-bound gate — HARD go/no-go for the two-stage move (ADR 0021 §8)

Open THIS notebook in Cursor, connect the kernel to your **A100** (kernel selector → Colab → **Auto Connect**), then **Run All** top to bottom. It scores the CURRENT one-stage adapter on the P1 unit-aware holdout **twice** — once conditioned on the free-text `instruction` (baseline), once on the ground-truth `attribute_spec_text` (oracle) — and prints:

```
METRIC_baseline=<float>
METRIC_oracle=<float>
{"oracle_gate": {…}}
```

**Paste back**: the two `METRIC_*` lines **and** the `{"oracle_gate": …}` JSON line.

**Gate**: PASS ⇔ `oracle ≥ baseline` (the ground-truth spec is at least as good a conditioner as free text → the semantic-IR seam is not lossy → proceed to P5/P6). If it FAILS, the two-stage move is off — stop and report.

Case trap: code lives at `/content/SLM` (UPPERCASE); the staged corpus + `SLM_ARTIFACT_ROOT=/content/slm` (lowercase). One wrong case silently skips every image.

In [ ]:
## CELL 1 — provision the runtime (idempotent; run first)
import os, pathlib, subprocess, sys

REPO = '/content/SLM'
BRANCH = 'feat/two-stage'   # the two-stage code (attribute_spec, oracle_gate) lives here
if not pathlib.Path(REPO, '.git').is_dir():
    print('cloning -> /content/SLM ...')
    subprocess.run(['git', 'clone', 'https://github.com/ericrcwu001/SLM', REPO], check=True)
os.chdir(REPO)

if not os.environ.get('SLM_PROVISIONED'):
    subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', 'checkout', BRANCH], check=True)
    subprocess.run(['git', 'pull', 'origin', BRANCH], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[sft]'], check=True)
    os.environ['SLM_PROVISIONED'] = '1'
print('HEAD:', subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip())

# HF token for the PRIVATE dataset + Qwen + adapter download. Prefer an uploaded .env, else paste.
def _env_token(name):
    for _p in ('/content/SLM/.env', '/content/.env', '.env'):
        fp = pathlib.Path(_p)
        if fp.is_file():
            for _l in fp.read_text().splitlines():
                s = _l.strip()
                if s.startswith(name + '='):
                    return s.split('=', 1)[1].strip().strip('"').strip("'")
    return None
if not os.environ.get('HF_TOKEN'):
    tok = _env_token('HF_TOKEN')
    if not tok:
        import getpass
        tok = getpass.getpass('Paste HF_TOKEN (read scope OK): ').strip()
    os.environ['HF_TOKEN'] = tok
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))

# stage the corpus ONLY if missing (~9.85GB) — images + frozen tokenizer resolve to /content/slm.
os.environ['SLM_ARTIFACT_ROOT'] = '/content/slm'
if not pathlib.Path('/content/slm/tokenizer/final/model.pt').is_file():
    print('corpus missing -> staging ~9.85GB from hf://datasets/ericrcwu/LUT_SLM ...')
    subprocess.run(['slm_stage', 'stage', '--durable-root', 'hf://datasets/ericrcwu/LUT_SLM',
                    '--local-root', '/content/slm'], check=True)
else:
    print('corpus already staged at /content/slm')
print('contents:', sorted(os.listdir('/content/slm')) if pathlib.Path('/content/slm').is_dir() else 'MISSING')

In [ ]:
## CELL 2 — imports + pre-run asserts
import os, pathlib, shutil
import torch
import bitsandbytes as bnb, peft, accelerate, qwen_vl_utils  # must all import
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor  # VL class must exist
os.environ['SLM_ARTIFACT_ROOT'] = '/content/slm'
assert torch.cuda.is_available(), 'no CUDA — wrong runtime (need A100)'
corpus = pathlib.Path('/content/slm')
assert corpus.is_dir(), 'staged corpus missing at /content/slm (run CELL 1)'
assert (corpus / 'tokenizer' / 'final' / 'model.pt').is_file(), 'frozen tokenizer weights missing'
free_gb = shutil.disk_usage('/content').free / 1e9
need_gb = 25 if not pathlib.Path('models/base_resized/vocab_resize_manifest.json').is_file() else 5
print('SLM_ARTIFACT_ROOT =', os.environ['SLM_ARTIFACT_ROOT'], '| free /content GB:', round(free_gb, 1))
assert free_gb > need_gb, f'need >{need_gb}GB free on /content (have {free_gb:.1f})'
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), '| pre-run asserts OK')

In [ ]:
## CELL 3 — build base_resized (reuse if present) + download the CURRENT one-stage adapter
import json, pathlib, os, subprocess, sys
D = pathlib.Path('models/base_resized')
if (D / 'vocab_resize_manifest.json').is_file() and (D / 'preprocessor_config.json').is_file():
    print('base_resized already built -> reusing (delete the dir to rebuild).')
else:
    subprocess.run([sys.executable, '-m', 'sft.vocab_resize', '--out', 'models/base_resized'],
                   check=True, env={**os.environ, 'SLM_ARTIFACT_ROOT': '/content/slm'})
m = json.load(open(D / 'vocab_resize_manifest.json'))
assert m.get('tokenizer_version') == 'vq_v2_srgbres_17to4_cb256_t64__w91cffdd2c82f', m.get('tokenizer_version')
assert m.get('vq_codebook_sha256'), 'null vq_codebook_sha256 — identity did not bind (check SLM_ARTIFACT_ROOT)'
print('base_resized OK; identity bound:', m['tokenizer_version'])

# The CURRENT one-stage adapter (ledger full-run winner). Download from HF if not already local.
ADAPTER_REPO = 'ericrcwu/LUT_SLM_sft_adapters'
ADAPTER_NAME = 'bl_a0ccbcff_smokefull'
ADAPTER_DIR = pathlib.Path('models/sft_adapters') / ADAPTER_NAME
if not (ADAPTER_DIR / 'adapter_config.json').is_file():
    from huggingface_hub import snapshot_download
    print(f'downloading adapter {ADAPTER_NAME} from {ADAPTER_REPO} ...')
    snapshot_download(repo_id=ADAPTER_REPO, allow_patterns=[f'{ADAPTER_NAME}/*'],
                      local_dir='models/sft_adapters', token=os.environ.get('HF_TOKEN'))
assert (ADAPTER_DIR / 'adapter_config.json').is_file(), f'adapter missing at {ADAPTER_DIR}'
print('adapter ready:', ADAPTER_DIR)

In [ ]:
## CELL 4 — run the oracle gate 🛑 paste back the two METRIC_* lines + the {"oracle_gate": …} JSON
!SLM_ARTIFACT_ROOT=/content/slm python -m sft.oracle_gate \
    --resized-model models/base_resized \
    --adapter models/sft_adapters/bl_a0ccbcff_smokefull \
    --limit 0
# --limit 0 = full P1 unit-aware holdout. Scores the SAME adapter under instruction (baseline) and
# ground-truth attribute_spec_text (oracle). PASS ⇔ METRIC_oracle >= METRIC_baseline.